# Labelers precision

We will attempt to calculate a measure of the labelers attempt using a consensus criteria for determining the "true label" per object. This criteria will be the same used for training as of writing, which is to select the mode as the true label (only between elements with no data quality comments.)

## Loading Label information

First, we must download the label informartion which is hosted on a google drive folder, and consists of several .csv files. We've saved the files on a local folder which is represented in the variable "path" of the next cell. Some excel duplicates where found and manually removed from the folder. Now, we can gather the desired "fields" or columns from all this files and merge them into a pandas dataframe.

In [4]:
# get relevant relative directories, create folder for storing any generated files. 

from pathlib import Path
import os

# Set current directory for using relative paths

current_dir =  Path(globals()['_dh'][0])
project_dir =  Path(globals()['_dh'][0]).parent

try:
    os.mkdir(os.path.join(current_dir, '1.0-jrb-alternative-approach'))
except:
    print("Failed to create folder. Perhaps the folder already exists, or there's a permissions issue.")

Failed to create folder. Perhaps the folder already exists, or there's a permissions issue.


In [6]:
import pandas as pd

path = os.path.join(project_dir, r'data/raw/label_data')
fields = ['File Name', 'MJD', 'Target ID', 'DB ID' ,'Classification', 'Data Quality']

raw_labels_df = pd.DataFrame()

# We import all of the csv files and merge them in star_df

for filename in os.listdir(path):
    file_path = os.path.join(path, filename)
    df = pd.read_csv (file_path, usecols=fields)
    df['Labeler'] = filename[:3]
    raw_labels_df = pd.concat([raw_labels_df, df], ignore_index=True)
print(raw_labels_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35253 entries, 0 to 35252
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Classification  35253 non-null  object
 1   DB ID           35253 non-null  int64 
 2   Data Quality    11158 non-null  object
 3   File Name       35253 non-null  object
 4   MJD             35253 non-null  int64 
 5   Target ID       35253 non-null  int64 
 6   Labeler         35253 non-null  object
dtypes: int64(3), object(4)
memory usage: 1.9+ MB
None


In [7]:
raw_labels_df

,Classification,DB ID,Data Quality,File Name,MJD,Target ID,Labeler
0,WDA,21899,NaN,spec-15249-59265-04601916346-21899.png,59265,4601916346,ARL
1,sdX,23360,BLEND,spec-15301-59338-04602344336-23360.png,59338,4602344336,ARL
2,WDA,18900,SNR,spec-15113-59217-04538755416-18900.png,59217,4538755416,ARL
3,WDA,18575,NaN,spec-15086-59267-04474143209-18575.png,59267,4474143209,ARL
4,WD,20528,SNR,spec-15198-59269-04351438385-20528.png,59269,4351438385,ARL
...,...,...,...,...,...,...,...
35248,WDA,15895,SNR,spec-15011-59222-04399876473-15895.png,59222,4399876473,VCH
35249,WDA,20605,NaN,spec-15200-59324-04589320996-20605.png,59324,4589320996,VCH
35250,WDA,19388,NaN,spec-15193-59243-04594923329-19388.png,59243,4594923329,VCH
35251,DUNNO,19161,SNR,spec-15121-59212-04342114705-19161.png,59212,4342114705,VCH


Let's fix some small issues (found when efirst exploring the dataset)

In [18]:
# Fiz some labels
raw_labels_df.loc[raw_labels_df['Classification'] == 'WELM', 'Classification'] = 'WDELM'
raw_labels_df.loc[raw_labels_df['Classification'] == 'DA', 'Classification'] = 'WDA'

# Replaces the data quality abscense of commentary with an 'OK'
raw_labels_df['Data Quality'] = raw_labels_df['Data Quality'].fillna('OK')
raw_labels_df.loc[raw_labels_df['Data Quality'] == ' ', 'Data Quality'] = 'OK'

In [19]:
raw_labels_df

,Classification,DB ID,Data Quality,File Name,MJD,Target ID,Labeler
0,WDA,21899,OK,spec-15249-59265-04601916346-21899.png,59265,4601916346,ARL
1,sdX,23360,BLEND,spec-15301-59338-04602344336-23360.png,59338,4602344336,ARL
2,WDA,18900,SNR,spec-15113-59217-04538755416-18900.png,59217,4538755416,ARL
3,WDA,18575,OK,spec-15086-59267-04474143209-18575.png,59267,4474143209,ARL
4,WD,20528,SNR,spec-15198-59269-04351438385-20528.png,59269,4351438385,ARL
...,...,...,...,...,...,...,...
35248,WDA,15895,SNR,spec-15011-59222-04399876473-15895.png,59222,4399876473,VCH
35249,WDA,20605,OK,spec-15200-59324-04589320996-20605.png,59324,4589320996,VCH
35250,WDA,19388,OK,spec-15193-59243-04594923329-19388.png,59243,4594923329,VCH
35251,DUNNO,19161,SNR,spec-15121-59212-04342114705-19161.png,59212,4342114705,VCH


And now we will drop observations with data quality issues.

In [23]:
qlabels_df = raw_labels_df.copy()
qlabels_df.drop(qlabels_df[qlabels_df['Data Quality'] != 'OK'].index, inplace = True)

In [24]:
qlabels_df

,Classification,DB ID,Data Quality,File Name,MJD,Target ID,Labeler
0,WDA,21899,OK,spec-15249-59265-04601916346-21899.png,59265,4601916346,ARL
3,WDA,18575,OK,spec-15086-59267-04474143209-18575.png,59267,4474143209,ARL
5,WDA,22243,OK,spec-15266-59306-04592699543-22243.png,59306,4592699543,ARL
6,WDA,19555,OK,spec-15165-59202-04545012720-19555.png,59202,4545012720,ARL
7,WD,20892,OK,spec-15221-59245-04306649216-20892.png,59245,4306649216,ARL
...,...,...,...,...,...,...,...
35242,STAR,20434,OK,spec-15288-59294-04552877469-20434.png,59294,4552877469,VCH
35244,STAR,20259,OK,spec-15195-59271-04543081849-20259.png,59271,4543081849,VCH
35246,STAR,20046,OK,spec-15285-59304-04550841319-20046.png,59304,4550841319,VCH
35249,WDA,20605,OK,spec-15200-59324-04589320996-20605.png,59324,4589320996,VCH


In [25]:
# Let's explore how many unique stellar objects we have left.

qlabels_df['Target ID'].duplicated(keep='first').value_counts()

True     16723
False     8113
Name: Target ID, dtype: int64

### Calculating the labelers precision as a whole

First we will see how many elements have a given number of repeated votes.

In [28]:
unique_ids = qlabels_df['Target ID'].unique()

print(unique_ids, len(unique_ids))

[4601916346 4474143209 4592699543 ... 4552859261 4347784790 4543221860] 8113


In [43]:
counts = dict()

for tid in unique_ids:
    number_of_labels = len(qlabels_df.loc[qlabels_df['Target ID'] == tid])

    # add key to dict or add count to number of elements with given number of labels.
    key = str(number_of_labels)
    if key in counts:
        counts[key] += 1
    else:
        counts[key] = 1
counts

{'4': 938,
 '1': 2247,
 '7': 252,
 '3': 1324,
 '5': 688,
 '2': 1886,
 '8': 152,
 '6': 430,
 '11': 18,
 '17': 1,
 '12': 13,
 '14': 7,
 '9': 89,
 '10': 41,
 '13': 8,
 '18': 3,
 '15': 9,
 '16': 5,
 '20': 2}

we can see elements with one or two labels are roughly half the entire set. We will discard them and use only objects with 3 or more observations.

In [31]:
qlabels_df.loc[qlabels_df['Target ID'] == 4553795050]

,Classification,DB ID,Data Quality,File Name,MJD,Target ID,Labeler
4234,WDB,19293,OK,spec-15127-59190-04553795050-19293.png,59190,4553795050,AXL
12013,WDC,19293,OK,spec-15190-59248-04553795050-19293.png,59248,4553795050,GAG
17274,WDA,19293,OK,spec-15190-59273-04553795050-19293.png,59273,4553795050,ISA
18200,WDB,19293,OK,spec-15127-59219-04553795050-19293.png,59219,4553795050,ISA
19215,WDC,19293,OK,spec-15190-59248-04553795050-19293.png,59248,4553795050,ISA
28327,WDB,19293,OK,spec-15127-59219-04553795050-19293.png,59219,4553795050,NLZ
33724,WDC,19293,OK,spec-15190-59273-04553795050-19293.png,59273,4553795050,SFA
34566,WDB,19293,OK,spec-15190-59267-04553795050-19293.png,59267,4553795050,VCH
34673,WDB,19293,OK,spec-15127-59190-04553795050-19293.png,59190,4553795050,VCH
35240,WDC,19293,OK,spec-15190-59248-04553795050-19293.png,59248,4553795050,VCH
